# Exploratory Data Analysis (EDA) and Statistics
This notebook is used for initial exploratory data analysis and statistical summaries of the dataset.

In [3]:
import sys
from pathlib import Path

# Force Python's engine to map modules to the project root
root_path = Path.cwd().parent
if str(root_path) not in sys.path:
    sys.path.insert(0, str(root_path))

import polars as pl
from src.config import seed_everything
from src.data_loader import load_raw_data, optimize_memory_usage

# Apply global seed for determinism
seed_everything()

print("Scanning directory and compiling wellbore matrix...")

# Pass the directory name "train" instead of a specific file name string
df = load_raw_data(directory_name="train")
df_optimized = optimize_memory_usage(df)

# Output profiling diagnostics
print("\n--- Ingestion Metrics ---")
print(f"Unified Data Matrix Shape: {df_optimized.shape}")
print(f"Unique Wellbore Folds Identified: {df_optimized['well_id'].n_unique()}")
print(f"RAM Allocation Footprint: {df_optimized.estimated_size('mb'):.2f} MB")

print("\n--- Compiled Target Attribute ('tvt') Summary ---")
# print(df_optimized.select(pl.col('tvt').describe())) 
# Option A: Call describe directly on a sliced DataFrame subset (Cleanest table output)
print(df_optimized.select("TVT").describe())

# Option B: Updated to use the correct uppercase "TVT" column name
print("\n--- Explicit Aggregations ---")
print(df_optimized.select(
    pl.col("TVT").min().alias("min"),
    pl.col("TVT").mean().alias("mean"),
    pl.col("TVT").max().alias("max"),
    pl.col("TVT").null_count().alias("nulls")
))

Scanning directory and compiling wellbore matrix...

--- Ingestion Metrics ---
Unified Data Matrix Shape: (5092255, 14)
Unique Wellbore Folds Identified: 773
RAM Allocation Footprint: 448.98 MB

--- Compiled Target Attribute ('tvt') Summary ---
shape: (9, 2)
┌────────────┬──────────────┐
│ statistic  ┆ TVT          │
│ ---        ┆ ---          │
│ str        ┆ f64          │
╞════════════╪══════════════╡
│ count      ┆ 5.092255e6   │
│ null_count ┆ 0.0          │
│ mean       ┆ 11503.643555 │
│ std        ┆ 639.971069   │
│ min        ┆ 9245.19043   │
│ 25%        ┆ 10987.929688 │
│ 50%        ┆ 11354.509766 │
│ 75%        ┆ 12038.259766 │
│ max        ┆ 12893.889648 │
└────────────┴──────────────┘

--- Explicit Aggregations ---
shape: (1, 4)
┌────────────┬──────────────┬──────────────┬───────┐
│ min        ┆ mean         ┆ max          ┆ nulls │
│ ---        ┆ ---          ┆ ---          ┆ ---   │
│ f32        ┆ f32          ┆ f32          ┆ u32   │
╞════════════╪══════════════╪═════

In [4]:
print(df_optimized.head(5))

shape: (5, 14)
┌─────────┬──────────┬───────────┬──────────┬───┬─────────────┬────────────┬────────────┬──────────┐
│ MD      ┆ X        ┆ Y         ┆ Z        ┆ … ┆ TVT         ┆ GR         ┆ TVT_input  ┆ well_id  │
│ ---     ┆ ---      ┆ ---       ┆ ---      ┆   ┆ ---         ┆ ---        ┆ ---        ┆ ---      │
│ f64     ┆ f64      ┆ f64       ┆ f64      ┆   ┆ f32         ┆ str        ┆ f32        ┆ str      │
╞═════════╪══════════╪═══════════╪══════════╪═══╪═════════════╪════════════╪════════════╪══════════╡
│ 10545.0 ┆ 2.9067e6 ┆ 1034714.9 ┆ -8510.12 ┆ … ┆ 10541.80957 ┆ 135.989659 ┆ 10541.8095 ┆ a7744f5b │
│         ┆          ┆           ┆          ┆   ┆             ┆ 25107527   ┆ 7          ┆          │
│ 10546.0 ┆ 2.9067e6 ┆ 1.0347e6  ┆ -8511.12 ┆ … ┆ 10542.80957 ┆ null       ┆ 10542.8095 ┆ a7744f5b │
│         ┆          ┆           ┆          ┆   ┆             ┆            ┆ 7          ┆          │
│ 10547.0 ┆ 2.9067e6 ┆ 1.0347e6  ┆ -8512.12 ┆ … ┆ 10543.80957 ┆ 132.293831 ┆